# Public data experiments
This is the notebook to run the public data experiment

In [1]:
import sys
from pathlib import Path

# Add the workspace root to sys.path
workspace_root = Path("/Users/eduardohome/workspace/benchmark_credit_scoring")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

# Now you can import from src
from src.utils.csv_config import read_csv_by_name, create_imbalanced_datasets, merge_xai_results
from src.model.functions import save_model, load_model, get_split_data
from src.xai.functions import get_train_iterations_shap, compute_xai_metrics_from_shap
from src.graph.functions import plot_model_boxplot, plot_imbalance_boxplot_grid

try:
    from src.model.functions import get_train_model
except Exception as exc:
    # provide helpful message if import fails
    print("Failed to import model utilities:", exc)
    print("This often means a missing dependency such as libomp for LightGBM.")
    print("On macOS you can install it with `brew install libomp` and then restart the kernel.")
    raise

# General imports
import pandas as pd
from pathlib import Path

In [2]:
# -------------------------
# Settings
# -------------------------
data_dir = Path("/Users/eduardohome/workspace/benchmark_credit_scoring/data")
#datasets_to_run = ["australian", "german", "japan", "kgccdea", "taiwan", "th02"] #kggmsc #taiwan
datasets_to_run = ["australian", "german", "japan", "kgccdea", "th02"] #kggmsc #taiwan
#datasets_to_run = ["australian"]
target_col = "target"

# Edit this list whenever you want more/less imbalance datasets
imbalance_levels = [10, 15, 20, 25, 30, 35, 40, 45, 50]
#imbalance_levels = [1, 2.5, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
n_iterations = 100
percentage = 90

# list of estimators understood by functions.get_model
model_choices = ["XGBoost", "LightGBM", "RandomForest", "AdaBoost-DT", "BoostDT", "BoostSGB", "BagNN", "ANN"] #, "SVM-Rbf"
#model_choices = ["XGBoost"] #, "SVM-Rbf"
product = "default"

## Interpretability metrics

In [3]:
# -------------------------
# Loop XAI metrics from saved SHAP
# -------------------------
for dataset_name in datasets_to_run:

    dataset_variants = [dataset_name] + [f"{dataset_name}_{lvl}" for lvl in imbalance_levels]

    out_dir = data_dir / "output" / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 90)
    print(f"BASE DATASET: {dataset_name}")
    print("#" * 90)

    for dataset_variant in dataset_variants:

        for model_choice in model_choices:

            print(f"Computing XAI metrics for {dataset_variant} | {model_choice}")

            # --------------------------------------------------
            # Load previously saved SHAP file
            # --------------------------------------------------
            shap_dir = out_dir / "xai" / f"{model_choice}" / f"{dataset_variant}" / "shap_values"
            shap_path = shap_dir / f"{dataset_variant}_{model_choice}_shap_values.csv"

            if not shap_path.exists():
                print(f"[SKIP] SHAP file not found: {shap_path}")
                continue

            df_shap = pd.read_csv(shap_path)

            # Optional sanity check
            required_cols = {"n_iteration", "index"}
            missing_cols = required_cols - set(df_shap.columns)
            if missing_cols:
                print(f"[SKIP] Missing required columns in SHAP file: {missing_cols}")
                continue

            # --------------------------------------------------
            # Compute metrics from saved SHAP
            # --------------------------------------------------
            (
                df_detail_local,
                df_resume_local,
                df_resume_global,
            ) = compute_xai_metrics_from_shap(df_shap)

            # --------------------------------------------------
            # Create output folders
            # --------------------------------------------------
            metrics_dir = out_dir / "xai" / f"{model_choice}" / f"{dataset_variant}" / "metrics"
            metrics_dir.mkdir(parents=True, exist_ok=True)

            # --------------------------------------------------
            # Save outputs
            # --------------------------------------------------
            df_detail_local.to_csv(
                metrics_dir / f"{dataset_variant}_{model_choice}_detail_local.csv",
                index=False
            )

            df_resume_local.to_csv(
                metrics_dir / f"{dataset_variant}_{model_choice}_resume_local.csv",
                index=False
            )

            df_resume_global.to_csv(
                metrics_dir / f"{dataset_variant}_{model_choice}_resume_global.csv",
                index=False
            )

            print(f"[OK] Saved metrics to {metrics_dir}")


##########################################################################################
BASE DATASET: australian
##########################################################################################
Computing XAI metrics for australian | XGBoost
[SKIP] SHAP file not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/xai/XGBoost/australian/shap_values/australian_XGBoost_shap_values.csv
Computing XAI metrics for australian | LightGBM
[SKIP] SHAP file not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/xai/LightGBM/australian/shap_values/australian_LightGBM_shap_values.csv
Computing XAI metrics for australian | RandomForest
[SKIP] SHAP file not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/xai/RandomForest/australian/shap_values/australian_RandomForest_shap_values.csv
Computing XAI metrics for australian | AdaBoost-DT
[SKIP] SHAP file not found: /Users/eduardohome/workspace/

## Summary results

In [4]:
results = merge_xai_results(
    data_dir=data_dir,
    datasets_to_run=datasets_to_run,
    imbalance_levels=imbalance_levels,
    model_choices=model_choices,
    file_types=["resume_local", "resume_global", "summary"],
    normalize=True,
)


Processing dataset: australian
  Merging file type: resume_local
Saved raw merged dataset: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/xai/australian_xai_all_models_resume_local.csv
Saved normalized merged dataset: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/xai/australian_xai_all_models_resume_local_normalized.csv
  Merging file type: resume_global
Saved raw merged dataset: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/xai/australian_xai_all_models_resume_global.csv
Saved normalized merged dataset: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/xai/australian_xai_all_models_resume_global_normalized.csv
  Merging file type: summary
Saved raw merged dataset: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/xai/australian_xai_all_models_summary.csv
Saved normalized merged dataset: /Users/eduardohome/workspace/benchmark_credit_scoring/